<a href="https://colab.research.google.com/github/shafifaris/Divisi-Data-IPOSS/blob/main/mpob_luas_tertanam_2023_2025_colab_FINAL_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scraper MPOB Luas Tertanam 2023-2025 - Versi Final

Notebook ini mengumpulkan data MPOB berikut secara otomatis:

- luas tertanam per negeri: matang, belum matang, dan total;
- tahun 2023, 2024, dan 2025.

Fokus default adalah halaman `Area by State` yang telah terbukti menyediakan tabel.
Tiga halaman kategori pemilik tidak dijalankan secara default karena pada halaman tersebut
pilihan tahun dapat tersedia tanpa tabel data. Pilihan ini mencegah hasil utama yang valid
gagal hanya karena halaman tambahan tidak memublikasikan tabel.

Struktur keluaran mengikuti Panduan Pengumpulan Data IPOSS: satu baris satu pengamatan,
nama kolom huruf kecil dengan garis bawah, satu kolom waktu `year`, angka murni,
serta kolom `sumber_link` dan `input_at` pada setiap baris.

**Sebelum menjalankan:** buat Colab Secrets bernama `MPOB_USERNAME` dan
`MPOB_PASSWORD`, lalu aktifkan *Notebook access*. Nilai rahasia tidak dicetak atau
disimpan di notebook. Apabila halaman MPOB terbuka tanpa login, notebook tetap berjalan.

Jalankan melalui **Runtime > Run all**. Ketika diminta, izinkan autentikasi Google
agar hasil dapat dikirim ke satu Google Sheet dan dibagikan kepada `dataiposs@gmail.com`.

## 1. Instalasi dan pemeriksaan browser

In [1]:
import os
import re
import shutil
import subprocess
import sys
import urllib.request
from pathlib import Path


def run_checked(command, label):
    print(f"[INFO] {label}...")
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        print(result.stdout[-6000:])
        raise RuntimeError(f"{label} gagal dengan exit code {result.returncode}.")
    return result.stdout.strip()


run_checked(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "selenium>=4.26",
        "beautifulsoup4",
        "lxml",
        "pytz",
        "gspread",
        "google-api-python-client",
    ],
    "Memasang library Python",
)


def find_chrome():
    candidates = [
        "google-chrome",
        "google-chrome-stable",
        "chromium",
        "chromium-browser",
    ]
    return next((shutil.which(name) for name in candidates if shutil.which(name)), None)


CHROME_BINARY = find_chrome()

# Colab umumnya sudah memiliki Google Chrome. Jika belum ada, pasang paket resmi.
if not CHROME_BINARY:
    deb_path = Path("/tmp/google-chrome-stable_current_amd64.deb")
    print("[INFO] Browser belum tersedia; mengunduh Google Chrome resmi...")
    urllib.request.urlretrieve(
        "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb",
        deb_path,
    )
    run_checked(["apt-get", "update", "-qq"], "Memperbarui indeks paket")
    run_checked(
        ["apt-get", "install", "-y", "-qq", str(deb_path)],
        "Memasang Google Chrome",
    )
    CHROME_BINARY = find_chrome()

if not CHROME_BINARY:
    raise RuntimeError(
        "Browser Chrome/Chromium tetap tidak ditemukan setelah instalasi. "
        "Pilih Runtime > Disconnect and delete runtime, lalu jalankan ulang notebook."
    )

version_result = run_checked([CHROME_BINARY, "--version"], "Memeriksa versi browser")

# Uji browser secara langsung agar cell tidak mencetak sukses palsu.
smoke = subprocess.run(
    [
        CHROME_BINARY,
        "--headless=new",
        "--no-sandbox",
        "--disable-dev-shm-usage",
        "--dump-dom",
        "about:blank",
    ],
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
if smoke.returncode != 0:
    print(smoke.stdout[-6000:])
    raise RuntimeError("Browser terpasang tetapi gagal dijalankan dalam mode headless.")

print(f"[OK] Browser siap: {CHROME_BINARY}")
print(f"[OK] Versi: {version_result}")

[INFO] Memasang library Python...
[INFO] Browser belum tersedia; mengunduh Google Chrome resmi...
[INFO] Memperbarui indeks paket...
[INFO] Memasang Google Chrome...
[INFO] Memeriksa versi browser...
[OK] Browser siap: /usr/bin/google-chrome
[OK] Versi: Google Chrome 153.0.8010.47


## 2. Import, Secrets, dan konfigurasi tetap

In [2]:
import io
import random
import tempfile
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import pytz
from IPython.display import display

from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select, WebDriverWait

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import userdata

    try:
        MPOB_USERNAME = userdata.get("MPOB_USERNAME")
    except Exception:
        MPOB_USERNAME = None
    try:
        MPOB_PASSWORD = userdata.get("MPOB_PASSWORD")
    except Exception:
        MPOB_PASSWORD = None
except ImportError:
    MPOB_USERNAME = None
    MPOB_PASSWORD = None

if MPOB_USERNAME and MPOB_PASSWORD:
    print("[OK] MPOB_USERNAME dan MPOB_PASSWORD terbaca dari Colab Secrets.")
else:
    print("[INFO] Secret login tidak tersedia. Login akan dilewati bila halaman terbuka.")

YEARS = [2023, 2024, 2025]
INPUT_AT = datetime.now(pytz.timezone("Asia/Jakarta")).strftime("%Y-%m-%d")
SHARE_EDITOR_EMAIL = "dataiposs@gmail.com"
SPREADSHEET_NAME = "mpob_luas_tertanam_2023_2025"
WORKSHEET_NAME = "data"
AUTO_UPLOAD_TO_GSHEET = True
SCRAPE_OPTIONAL_OWNERSHIP = False
DELAY_RANGE_SECONDS = (2.0, 4.0)

PAGE_CONFIG = {
    "state": {
        "url": "https://prestasisawit.mpob.gov.my/en/plantations-area",
        "label": "Area by State",
        "breakdown_type": "state",
        "maturity_status": None,
    },
    "total_planted": {
        "url": "https://prestasisawit.mpob.gov.my/en/plantations-planted",
        "label": "Ownership - Planted Area",
        "breakdown_type": "ownership",
        "maturity_status": "total_planted",
    },
    "matured": {
        "url": "https://prestasisawit.mpob.gov.my/en/plantations-matured",
        "label": "Ownership - Matured Area",
        "breakdown_type": "ownership",
        "maturity_status": "matured",
    },
    "immature": {
        "url": "https://prestasisawit.mpob.gov.my/en/plantations-immature",
        "label": "Ownership - Immature Area",
        "breakdown_type": "ownership",
        "maturity_status": "immature",
    },
}

ACTIVE_PAGE_KEYS = ["state"]
if SCRAPE_OPTIONAL_OWNERSHIP:
    ACTIVE_PAGE_KEYS.extend(["total_planted", "matured", "immature"])

OUTPUT_MAIN = "mpob_luas_tertanam_2023_2025.csv"
OUTPUT_QC = "mpob_luas_tertanam_qc_2023_2025.csv"
OUTPUT_SAMPLE = "mpob_luas_tertanam_sampel_qc_2023_2025.csv"

SCRAPE_LOG = []
RAW_RECORDS = []


def log(level, message):
    line = f"[{level}] {message}"
    SCRAPE_LOG.append(line)
    print(line)


def clean_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def normalize_name(value):
    value = clean_text(value).lower().replace("%", " percentage ")
    return re.sub(r"[^a-z0-9]+", "_", value).strip("_")


print(f"[OK] Tanggal input Asia/Jakarta: {INPUT_AT}")
print(f"[OK] Tahun target: {YEARS}")

[OK] MPOB_USERNAME dan MPOB_PASSWORD terbaca dari Colab Secrets.
[OK] Tanggal input Asia/Jakarta: 2026-09-17
[OK] Tahun target: [2023, 2024, 2025]


## 3. Browser Selenium yang tahan terhadap variasi Colab

In [3]:
_BROWSER_PROFILE_DIRS = []


def build_options(headless_argument="--headless=new"):
    profile_dir = tempfile.mkdtemp(prefix="mpob_chrome_")
    _BROWSER_PROFILE_DIRS.append(profile_dir)

    options = webdriver.ChromeOptions()
    options.binary_location = CHROME_BINARY
    for argument in [
        headless_argument,
        "--no-sandbox",
        "--disable-setuid-sandbox",
        "--disable-dev-shm-usage",
        "--disable-gpu",
        "--disable-software-rasterizer",
        "--remote-debugging-pipe",
        "--window-size=1920,1080",
        "--lang=en-US",
        "--disable-background-networking",
        "--disable-default-apps",
        "--disable-extensions",
        "--no-first-run",
        f"--user-data-dir={profile_dir}",
    ]:
        options.add_argument(argument)
    return options


def major_version(command):
    try:
        text = subprocess.check_output(command, text=True, stderr=subprocess.STDOUT)
        match = re.search(r"(\d+)\.", text)
        return int(match.group(1)) if match else None
    except Exception:
        return None


def make_driver():
    if not CHROME_BINARY or not Path(CHROME_BINARY).exists():
        raise RuntimeError(f"Binary browser tidak valid: {CHROME_BINARY}")

    chrome_major = major_version([CHROME_BINARY, "--version"])
    local_driver = shutil.which("chromedriver")
    driver_major = major_version([local_driver, "--version"]) if local_driver else None

    print(f"[INFO] Browser: {CHROME_BINARY} (major={chrome_major})")
    if local_driver:
        print(f"[INFO] ChromeDriver lokal: {local_driver} (major={driver_major})")
    else:
        print("[INFO] ChromeDriver lokal tidak ada; Selenium Manager akan memilih driver.")

    attempts = []
    for headless_argument in ["--headless=new", "--headless"]:
        # Prioritaskan Selenium Manager agar driver mengikuti versi browser.
        attempts.append((f"Selenium Manager, {headless_argument}", None, headless_argument))
        # Driver lokal hanya dicoba bila versi mayor cocok.
        if local_driver and chrome_major and driver_major == chrome_major:
            attempts.append((f"driver lokal, {headless_argument}", local_driver, headless_argument))

    errors = []
    for label, driver_path, headless_argument in attempts:
        try:
            options = build_options(headless_argument)
            service = (
                Service(executable_path=driver_path, log_output="/tmp/chromedriver.log")
                if driver_path
                else Service(log_output="/tmp/chromedriver.log")
            )
            driver = webdriver.Chrome(service=service, options=options)
            driver.set_page_load_timeout(90)
            driver.set_script_timeout(60)
            print(f"[OK] Selenium berhasil dimulai memakai {label}.")
            return driver
        except WebDriverException as exc:
            errors.append(f"{label}: {type(exc).__name__}: {str(exc)[:1200]}")

    try:
        driver_log = Path("/tmp/chromedriver.log").read_text(errors="replace")[-5000:]
    except Exception:
        driver_log = "Log ChromeDriver tidak tersedia."

    raise RuntimeError(
        "Semua strategi memulai Selenium gagal.\n"
        + "\n".join(errors)
        + "\n\nLog ChromeDriver terakhir:\n"
        + driver_log
    )


def test_driver():
    driver = make_driver()
    try:
        driver.get("data:text/html,<title>Selenium OK</title><h1>OK</h1>")
        if driver.title != "Selenium OK":
            raise RuntimeError("Browser menyala tetapi halaman uji tidak terbaca.")
        print(f"[OK] Uji Selenium PASS. Browser version: {driver.capabilities.get('browserVersion')}")
    finally:
        driver.quit()


test_driver()

[INFO] Browser: /usr/bin/google-chrome (major=153)
[INFO] ChromeDriver lokal tidak ada; Selenium Manager akan memilih driver.
[OK] Selenium berhasil dimulai memakai Selenium Manager, --headless=new.
[OK] Uji Selenium PASS. Browser version: 153.0.8010.47


## 4. Login opsional dan pemilihan tahun

In [4]:
USERNAME_SELECTORS = [
    "input[name='username']",
    "input[id='username']",
    "input[name*='user' i]",
    "input[id*='user' i]",
    "input[type='email']",
    "input[name*='email' i]",
]
PASSWORD_SELECTORS = ["input[type='password']"]
SUBMIT_SELECTORS = [
    "button[type='submit']",
    "input[type='submit']",
    "button[name*='login' i]",
    "button[id*='login' i]",
]


def first_displayed(driver, selectors, require_enabled=True):
    for selector in selectors:
        for element in driver.find_elements(By.CSS_SELECTOR, selector):
            try:
                if element.is_displayed() and (element.is_enabled() or not require_enabled):
                    return element
            except Exception:
                continue
    return None


def body_text(driver):
    try:
        return clean_text(driver.find_element(By.TAG_NAME, "body").text)
    except Exception:
        return ""


def login_if_present(driver, return_url):
    password_field = first_displayed(driver, PASSWORD_SELECTORS)
    if password_field is None:
        return False

    lowered = body_text(driver).lower()
    if any(word in lowered for word in ["captcha", "verify you are human", "robot"]):
        raise RuntimeError(
            "MPOB menampilkan verifikasi manusia/CAPTCHA. Notebook berhenti dan tidak mencoba melewatinya."
        )
    if not (MPOB_USERNAME and MPOB_PASSWORD):
        raise RuntimeError(
            "Form login muncul, tetapi MPOB_USERNAME atau MPOB_PASSWORD belum tersedia "
            "di Colab Secrets atau Notebook access belum diaktifkan."
        )

    username_field = first_displayed(driver, USERNAME_SELECTORS)
    submit_button = first_displayed(driver, SUBMIT_SELECTORS)
    if username_field is None or submit_button is None:
        raise RuntimeError(
            "Form login ditemukan, tetapi kolom username atau tombol masuk tidak dikenali."
        )

    old_url = driver.current_url
    username_field.clear()
    username_field.send_keys(MPOB_USERNAME)
    password_field.clear()
    password_field.send_keys(MPOB_PASSWORD)
    submit_button.click()

    try:
        WebDriverWait(driver, 60, poll_frequency=0.5).until(
            lambda d: first_displayed(d, PASSWORD_SELECTORS) is None
            or d.current_url != old_url
        )
    except TimeoutException as exc:
        raise RuntimeError("Proses login tidak selesai dalam 60 detik.") from exc

    time.sleep(1.5)
    failure_text = body_text(driver).lower()
    if first_displayed(driver, PASSWORD_SELECTORS) is not None or any(
        word in failure_text
        for word in ["invalid", "incorrect", "login failed", "authentication failed"]
    ):
        raise RuntimeError("Login gagal. Periksa isi Colab Secrets dan akses notebook.")

    log("OK", "Login MPOB berhasil; kredensial tidak dicetak.")
    driver.get(return_url)
    WebDriverWait(driver, 60).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    return True


def table_signature(driver):
    parts = []
    for table in driver.find_elements(By.CSS_SELECTOR, "table"):
        try:
            parts.append(clean_text(table.get_attribute("innerText"))[:3000])
        except Exception:
            continue
    return "||".join(parts)


def locate_year_select(driver, year):
    candidates = []
    for element in driver.find_elements(By.TAG_NAME, "select"):
        try:
            select = Select(element)
            options = [
                (clean_text(opt.text), clean_text(opt.get_attribute("value")))
                for opt in select.options
            ]
            if any(re.search(rf"\b{year}\b", f"{label} {value}") for label, value in options):
                attributes = " ".join(
                    clean_text(element.get_attribute(name))
                    for name in ["id", "name", "class", "aria-label"]
                ).lower()
                score = 2 if any(word in attributes for word in ["year", "tahun", "yr"]) else 1
                candidates.append((score, element))
        except Exception:
            continue
    if not candidates:
        raise RuntimeError(f"Dropdown yang memuat tahun {year} tidak ditemukan.")
    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1]


def select_year(driver, year):
    element = locate_year_select(driver, year)
    select = Select(element)
    before = table_signature(driver)

    selected_value = None
    selected_label = None
    selected_index = None
    for index, option in enumerate(select.options):
        label = clean_text(option.text)
        value = clean_text(option.get_attribute("value"))
        if re.search(rf"\b{year}\b", f"{label} {value}"):
            selected_value = value
            selected_label = label
            selected_index = index
            break
    if selected_value is None and selected_label is None:
        raise RuntimeError(f"Pilihan tahun {year} tidak tersedia.")

    try:
        if selected_value:
            select.select_by_value(selected_value)
        else:
            select.select_by_visible_text(selected_label)
    except Exception:
        # Fallback untuk native select yang disembunyikan oleh plugin UI.
        driver.execute_script(
            "arguments[0].selectedIndex=arguments[1];"
            "arguments[0].dispatchEvent(new Event('input',{bubbles:true}));"
            "arguments[0].dispatchEvent(new Event('change',{bubbles:true}));",
            element,
            selected_index,
        )

    def table_ready(drv):
        tables = drv.find_elements(By.CSS_SELECTOR, "table")
        real_table = any(len(t.find_elements(By.CSS_SELECTOR, "tr")) >= 2 for t in tables)
        current = table_signature(drv)
        return real_table and bool(current.strip()) and (current != before or bool(before.strip()))

    try:
        WebDriverWait(driver, 75, poll_frequency=0.5).until(table_ready)
    except TimeoutException as exc:
        raise RuntimeError(f"Tabel tahun {year} tidak muncul dalam 75 detik.") from exc

    time.sleep(1.5)
    log("OK", f"Tahun {year} berhasil dipilih.")

## 5. Parser tabel MPOB

In [5]:
def flatten_columns(columns):
    if isinstance(columns, pd.MultiIndex):
        result = []
        for values in columns:
            parts = [clean_text(v) for v in values if clean_text(v).lower() != "nan"]
            result.append(normalize_name(" ".join(dict.fromkeys(parts))))
        return result
    return [normalize_name(value) for value in columns]


def number_or_nan(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    text = clean_text(value)
    if text.lower() in {"", "nan", "na", "n/a", "null", "-", "--"}:
        return np.nan
    text = re.sub(r"(?<=\d),(?=\d{3}(?:\D|$))", "", text)
    text = text.replace(" ", "")
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    return pd.to_numeric(match.group(0), errors="coerce") if match else np.nan


def html_tables_from_driver(driver):
    frames = []
    for table in driver.find_elements(By.CSS_SELECTOR, "table"):
        try:
            html = table.get_attribute("outerHTML")
            for frame in pd.read_html(io.StringIO(html), displayed_only=False):
                frame.columns = flatten_columns(frame.columns)
                frame = frame.dropna(how="all").reset_index(drop=True)
                if not frame.empty and len(frame.columns) >= 2:
                    frames.append(frame)
        except Exception:
            continue
    if not frames:
        raise RuntimeError("Elemen tabel ada, tetapi tidak dapat dibaca sebagai tabel data.")
    return frames


def find_column(columns, include, exclude=()):
    for column in columns:
        name = normalize_name(column)
        if any(term in name for term in include) and not any(term in name for term in exclude):
            return column
    return None


def geography_level(state):
    name = normalize_name(state)
    if name in {"malaysia", "total", "grand_total", "all_malaysia"} or "malaysia_total" in name:
        return "country"
    if "peninsular" in name or "semenanjung" in name:
        return "region"
    return "state"


def parse_state_tables(tables, year, source_url):
    diagnostics = []
    for table_index, frame in enumerate(tables):
        columns = list(frame.columns)
        state_col = find_column(columns, ["state", "negeri"])
        matured_col = find_column(columns, ["matured", "mature"], ["immature"])
        immature_col = find_column(columns, ["immature", "not_mature"])
        total_col = find_column(columns, ["total", "planted"], ["percentage", "percent"])
        diagnostics.append(columns)
        if not all([state_col, matured_col, immature_col, total_col]):
            continue

        records = []
        for _, row in frame.iterrows():
            state = clean_text(row.get(state_col))
            if not state or normalize_name(state) in {"state", "negeri"}:
                continue
            values = {
                "matured": number_or_nan(row.get(matured_col)),
                "immature": number_or_nan(row.get(immature_col)),
                "total_planted": number_or_nan(row.get(total_col)),
            }
            if all(pd.isna(value) for value in values.values()):
                continue
            level = geography_level(state)
            for status, area in values.items():
                records.append(
                    {
                        "year": int(year),
                        "breakdown_type": "state",
                        "geography_level": level,
                        "state": state,
                        "ownership_category": "",
                        "maturity_status": status,
                        "area_hectares": area,
                        "sumber_link": source_url,
                        "input_at": INPUT_AT,
                        "catatan": (
                            f"Baris agregat tingkat {level} dari sumber MPOB."
                            if level != "state"
                            else ""
                        ),
                    }
                )
        if records:
            return records, frame

    raise RuntimeError(
        "Tabel negeri dengan kolom State/Matured/Immature/Total tidak ditemukan. "
        f"Header yang terbaca: {diagnostics}"
    )


def parse_ownership_tables(tables, year, source_url, maturity_status):
    diagnostics = []
    label_terms = ["ownership", "owner", "category", "kategori", "sector", "type", "holding"]
    value_terms = [maturity_status, "area", "hectare", "hectares", "ha", "total", "planted"]

    for table_index, frame in enumerate(tables):
        columns = list(frame.columns)
        label_col = find_column(columns, label_terms)

        if label_col is None:
            for column in columns:
                sample = frame[column].dropna().astype(str).head(12)
                if not sample.empty and sample.str.contains(r"[A-Za-z]", regex=True).mean() >= 0.6:
                    label_col = column
                    break

        candidates = []
        for column in columns:
            name = normalize_name(column)
            if column == label_col or any(term in name for term in ["percent", "percentage", "share"]):
                continue
            converted = frame[column].map(number_or_nan)
            valid = converted.dropna()
            if valid.empty or converted.notna().mean() < 0.35:
                continue
            keyword_score = sum(term in name for term in value_terms)
            magnitude_score = float(valid.abs().median() >= 100)
            candidates.append((keyword_score, magnitude_score, converted.notna().mean(), column))

        candidates.sort(reverse=True)
        value_col = candidates[0][3] if candidates else None
        diagnostics.append(
            {"columns": columns, "label_col": label_col, "value_col": value_col}
        )
        if label_col is None or value_col is None:
            continue

        records = []
        for _, row in frame.iterrows():
            category = clean_text(row.get(label_col))
            area = number_or_nan(row.get(value_col))
            if not category or pd.isna(area):
                continue
            if normalize_name(category) == normalize_name(label_col):
                continue
            records.append(
                {
                    "year": int(year),
                    "breakdown_type": "ownership",
                    "geography_level": "country",
                    "state": "",
                    "ownership_category": category,
                    "maturity_status": maturity_status,
                    "area_hectares": area,
                    "sumber_link": source_url,
                    "input_at": INPUT_AT,
                    "catatan": (
                        "Kategori agregat dipertahankan sesuai sumber MPOB."
                        if normalize_name(category) in {"total", "malaysia", "grand_total"}
                        else ""
                    ),
                }
            )
        if records:
            return records, frame

    raise RuntimeError(
        "Tabel kategori pemilik tidak dapat dikenali. "
        f"Diagnostik: {diagnostics}"
    )


def scrape_page_year(driver, page_key, year):
    config = PAGE_CONFIG[page_key]
    driver.get(config["url"])
    WebDriverWait(driver, 60).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    login_if_present(driver, config["url"])

    if first_displayed(driver, PASSWORD_SELECTORS) is not None:
        raise RuntimeError("Halaman data masih meminta login setelah proses autentikasi.")

    select_year(driver, year)
    tables = html_tables_from_driver(driver)
    resolved_url = driver.current_url

    if page_key == "state":
        records, source_table = parse_state_tables(tables, year, resolved_url)
    else:
        records, source_table = parse_ownership_tables(
            tables,
            year,
            resolved_url,
            config["maturity_status"],
        )

    return records, source_table

## 6. Scraping iteratif 2023-2025

In [6]:
RAW_RECORDS.clear()
SCRAPE_LOG.clear()
successful_pairs = set()
failures = []

driver = make_driver()
try:
    for page_key in ACTIVE_PAGE_KEYS:
        config = PAGE_CONFIG[page_key]
        for year in YEARS:
            print(f"\n[INFO] Mengambil {config['label']} - {year}")
            try:
                records, source_table = scrape_page_year(driver, page_key, year)
                RAW_RECORDS.extend(records)
                successful_pairs.add((page_key, year))
                log(
                    "OK",
                    f"{config['label']} {year}: {len(records)} baris format panjang "
                    f"dari {len(source_table)} baris tabel sumber.",
                )
            except Exception as exc:
                message = f"{config['label']} {year}: {type(exc).__name__}: {exc}"
                failures.append(message)
                log("ERROR", message)
            time.sleep(random.uniform(*DELAY_RANGE_SECONDS))
finally:
    driver.quit()
    print("[OK] Browser Selenium ditutup.")

expected_pairs = {(page_key, year) for page_key in ACTIVE_PAGE_KEYS for year in YEARS}
missing_pairs = sorted(expected_pairs - successful_pairs)
if missing_pairs:
    raise RuntimeError(
        "SCRAPING BELUM LENGKAP. Tidak ada file final atau Google Sheet yang dibuat.\n"
        f"Pasangan halaman-tahun gagal: {missing_pairs}\n"
        + "\n".join(failures)
    )

print(f"[OK] Seluruh {len(expected_pairs)} pasangan halaman-tahun berhasil.")
print(f"[OK] Total baris mentah format panjang: {len(RAW_RECORDS):,}")

[INFO] Browser: /usr/bin/google-chrome (major=153)
[INFO] ChromeDriver lokal tidak ada; Selenium Manager akan memilih driver.
[OK] Selenium berhasil dimulai memakai Selenium Manager, --headless=new.

[INFO] Mengambil Area by State - 2023
[OK] Tahun 2023 berhasil dipilih.
[OK] Area by State 2023: 48 baris format panjang dari 16 baris tabel sumber.

[INFO] Mengambil Area by State - 2024
[OK] Tahun 2024 berhasil dipilih.
[OK] Area by State 2024: 48 baris format panjang dari 16 baris tabel sumber.

[INFO] Mengambil Area by State - 2025
[OK] Tahun 2025 berhasil dipilih.
[OK] Area by State 2025: 48 baris format panjang dari 16 baris tabel sumber.
[OK] Browser Selenium ditutup.
[OK] Seluruh 3 pasangan halaman-tahun berhasil.
[OK] Total baris mentah format panjang: 144


## 7. Pembersihan dan quality control

In [10]:
# ============================================================
# CELL 7 — TRANSFORMASI DAN QUALITY CONTROL
# ============================================================

data_final = pd.DataFrame(RAW_RECORDS)

if data_final.empty:
    raise RuntimeError(
        "RAW_RECORDS kosong. Jalankan kembali Cell 6."
    )

# Ambil hanya data Area by State.
if "breakdown_type" in data_final.columns:
    data_final = data_final[
        data_final["breakdown_type"].eq("state")
    ].copy()

if data_final.empty:
    raise RuntimeError(
        "Data Area by State tidak ditemukan."
    )

# Nama ini lebih tepat karena mencakup
# matured, immature, dan total_planted.
if "maturity_status" in data_final.columns:
    data_final = data_final.rename(
        columns={"maturity_status": "area_type"}
    )

# Kolom yang harus tersedia dari hasil scraping.
required_columns = [
    "year",
    "state",
    "area_type",
    "area_hectares",
    "sumber_link",
    "input_at",
]

missing_columns = [
    column
    for column in required_columns
    if column not in data_final.columns
]

if missing_columns:
    raise RuntimeError(
        f"Kolom hasil scraping belum lengkap: {missing_columns}"
    )

# Kolom final untuk Google Sheet/database/dashboard.
FINAL_COLUMNS = [
    "year",
    "state",
    "area_type",
    "area_hectares",
    "sumber_link",
    "input_at",
]

data_final = data_final[FINAL_COLUMNS].copy()

# Membersihkan kolom teks.
text_columns = [
    "state",
    "area_type",
    "sumber_link",
    "input_at",
]

for column in text_columns:
    data_final[column] = (
        data_final[column]
        .fillna("")
        .astype(str)
        .map(clean_text)
    )

# Memastikan tipe data benar.
data_final["year"] = pd.to_numeric(
    data_final["year"],
    errors="raise",
).astype("int64")

data_final["area_hectares"] = pd.to_numeric(
    data_final["area_hectares"],
    errors="coerce",
)

# Menghapus baris agregat karena tugas meminta per negeri.
aggregate_rows = {
    "malaysia",
    "peninsular_malaysia",
    "sabah_sarawak",
}

normalized_state = data_final["state"].map(
    normalize_name
)

data_final = (
    data_final[
        ~normalized_state.isin(aggregate_rows)
    ]
    .copy()
    .reset_index(drop=True)
)

if data_final.empty:
    raise RuntimeError(
        "Data kosong setelah baris agregat dihapus."
    )

# Kunci unik setiap pengamatan.
BUSINESS_KEY = [
    "year",
    "state",
    "area_type",
]

data_final = (
    data_final
    .sort_values(BUSINESS_KEY, kind="stable")
    .reset_index(drop=True)
)

# ============================================================
# VALIDASI STRUKTUR IPOSS
# ============================================================

# Nama kolom harus lowercase_snake_case.
if list(data_final.columns) != [
    normalize_name(column)
    for column in data_final.columns
]:
    raise AssertionError(
        "Nama kolom belum seluruhnya lowercase_snake_case."
    )

# Tahun harus tepat 2023–2025.
obtained_years = sorted(
    data_final["year"].unique().tolist()
)

if obtained_years != YEARS:
    raise AssertionError(
        f"Tahun tidak lengkap. "
        f"Diperoleh={obtained_years}, "
        f"seharusnya={YEARS}."
    )

# Memeriksa format input_at.
valid_input_at = data_final[
    "input_at"
].str.fullmatch(r"\d{4}-\d{2}-\d{2}")

if not valid_input_at.all():
    display(
        data_final[
            ~valid_input_at
        ]
    )

    raise AssertionError(
        "Ada input_at yang tidak berformat YYYY-mm-dd."
    )

# Memeriksa sumber_link.
if data_final["sumber_link"].eq("").any():
    display(
        data_final[
            data_final["sumber_link"].eq("")
        ]
    )

    raise AssertionError(
        "Ada baris tanpa sumber_link."
    )

# Memeriksa nama negeri.
if data_final["state"].eq("").any():
    display(
        data_final[
            data_final["state"].eq("")
        ]
    )

    raise AssertionError(
        "Ada baris tanpa nama negeri."
    )

# Memeriksa nilai luas kosong.
if data_final["area_hectares"].isna().any():
    display(
        data_final[
            data_final["area_hectares"].isna()
        ]
    )

    raise AssertionError(
        "Ada nilai area_hectares yang tidak terbaca."
    )

# Memeriksa luas negatif.
if (data_final["area_hectares"] < 0).any():
    display(
        data_final[
            data_final["area_hectares"] < 0
        ]
    )

    raise AssertionError(
        "Ada nilai area_hectares negatif."
    )

# ============================================================
# VALIDASI JENIS LUAS
# ============================================================

valid_area_types = {
    "matured",
    "immature",
    "total_planted",
}

obtained_area_types = set(
    data_final["area_type"].unique()
)

invalid_area_types = (
    obtained_area_types
    - valid_area_types
)

if invalid_area_types:
    raise AssertionError(
        f"Jenis luas tidak dikenali: {invalid_area_types}"
    )

missing_area_types = (
    valid_area_types
    - obtained_area_types
)

if missing_area_types:
    raise AssertionError(
        f"Jenis luas belum lengkap: {missing_area_types}"
    )

# ============================================================
# VALIDASI DUPLIKAT
# ============================================================

duplicate_mask = data_final.duplicated(
    BUSINESS_KEY,
    keep=False,
)

if duplicate_mask.any():
    display(
        data_final.loc[
            duplicate_mask,
            BUSINESS_KEY + ["area_hectares"],
        ]
    )

    raise AssertionError(
        "Ada duplikat berdasarkan year, state, dan area_type."
    )

# ============================================================
# MEMBUAT TABEL QC
# ============================================================

state_pivot = (
    data_final.pivot(
        index=[
            "year",
            "state",
        ],
        columns="area_type",
        values="area_hectares",
    )
    .reset_index()
)

state_pivot.columns.name = None

for column in [
    "matured",
    "immature",
    "total_planted",
]:
    if column not in state_pivot.columns:
        state_pivot[column] = np.nan

# QC: matured + immature = total_planted.
state_pivot["calculated_total"] = (
    state_pivot["matured"]
    + state_pivot["immature"]
)

state_pivot["difference"] = (
    state_pivot["total_planted"]
    - state_pivot["calculated_total"]
)

complete_values = state_pivot[
    [
        "matured",
        "immature",
        "total_planted",
    ]
].notna().all(axis=1)

state_pivot["qc_status"] = np.where(
    complete_values
    & state_pivot["difference"].abs().le(1),
    "PASS",
    np.where(
        complete_values,
        "FAIL",
        "INCOMPLETE",
    ),
)

failed_qc = state_pivot[
    state_pivot["qc_status"].ne("PASS")
]

if not failed_qc.empty:
    display(failed_qc)

    raise AssertionError(
        "QC gagal: matured + immature "
        "tidak sama dengan total_planted."
    )

# ============================================================
# VALIDASI JUMLAH NEGERI DAN BARIS
# ============================================================

state_counts = (
    data_final
    .groupby("year")["state"]
    .nunique()
)

incorrect_state_counts = {
    year: int(state_counts.get(year, 0))
    for year in YEARS
    if state_counts.get(year, 0) != 13
}

if incorrect_state_counts:
    raise AssertionError(
        f"Jumlah negeri tidak sesuai: "
        f"{incorrect_state_counts}. "
        "Seharusnya 13 negeri per tahun."
    )

# Setiap negeri-tahun harus memiliki tiga jenis luas.
area_types_per_state = (
    data_final
    .groupby(
        [
            "year",
            "state",
        ]
    )["area_type"]
    .nunique()
)

incomplete_state = area_types_per_state[
    area_types_per_state.ne(3)
]

if not incomplete_state.empty:
    display(
        incomplete_state
        .rename("jumlah_area_type")
        .reset_index()
    )

    raise AssertionError(
        "Ada negeri-tahun yang tidak memiliki "
        "matured, immature, dan total_planted."
    )

# Setiap tahun: 13 negeri × 3 jenis luas = 39 baris.
rows_per_year = (
    data_final
    .groupby("year")
    .size()
)

incorrect_rows = {
    year: int(rows_per_year.get(year, 0))
    for year in YEARS
    if rows_per_year.get(year, 0) != 39
}

if incorrect_rows:
    raise AssertionError(
        f"Jumlah baris per tahun tidak sesuai: "
        f"{incorrect_rows}. "
        "Seharusnya 39 baris per tahun."
    )

# Total: 13 negeri × 3 jenis luas × 3 tahun.
if len(data_final) != 117:
    raise AssertionError(
        f"Jumlah baris final {len(data_final)}, "
        "seharusnya 117."
    )

# ============================================================
# HASIL QC DAN SAMPEL MANUAL
# ============================================================

qc_final = state_pivot.copy()

qc_sample = (
    data_final
    .sample(
        n=min(10, len(data_final)),
        random_state=2026,
    )
    .sort_values(BUSINESS_KEY)
    .reset_index(drop=True)
)

DATASET_LOLOS_QC = True

print(
    f"[OK] QC PASS. Jumlah baris final: "
    f"{len(data_final):,}"
)

print("\nJumlah negeri per tahun:")
display(
    state_counts
    .rename("jumlah_negeri")
    .reset_index()
)

print("\nJumlah baris per tahun:")
display(
    rows_per_year
    .rename("jumlah_baris")
    .reset_index()
)

print("\nRingkasan QC:")
display(
    state_pivot
    .groupby(
        [
            "year",
            "qc_status",
        ]
    )
    .size()
    .rename("jumlah")
    .reset_index()
)

print("\nContoh data final:")
display(
    data_final.head(12)
)

print("\nSampel QC manual:")
display(qc_sample)

[OK] QC PASS. Jumlah baris final: 117

Jumlah negeri per tahun:


,year,jumlah_negeri
0,2023,13
1,2024,13
2,2025,13



Jumlah baris per tahun:


,year,jumlah_baris
0,2023,39
1,2024,39
2,2025,39



Ringkasan QC:


,year,qc_status,jumlah
0,2023,PASS,13
1,2024,PASS,13
2,2025,PASS,13



Contoh data final:


,year,state,area_type,area_hectares,sumber_link,input_at
0,2023,Johor,immature,46493,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
1,2023,Johor,matured,624369,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
2,2023,Johor,total_planted,670862,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
3,2023,Kedah,immature,9369,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
4,2023,Kedah,matured,76502,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
5,2023,Kedah,total_planted,85871,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
6,2023,Kelantan,immature,17319,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
7,2023,Kelantan,matured,141322,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
8,2023,Kelantan,total_planted,158641,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
9,2023,Melaka,immature,3416,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17



Sampel QC manual:


,year,state,area_type,area_hectares,sumber_link,input_at
0,2023,Melaka,matured,47667,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
1,2024,Negeri Sembilan,matured,168021,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
2,2024,Perak,immature,30460,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
3,2024,Selangor,total_planted,102980,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
4,2024,Terengganu,immature,21711,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
5,2025,Johor,total_planted,675558,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
6,2025,Perlis,total_planted,927,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
7,2025,Sabah,immature,228559,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
8,2025,Sabah,matured,1267999,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
9,2025,Terengganu,immature,20124,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17


## 8. Simpan CSV dan unduh

In [11]:
if not DATASET_LOLOS_QC:
    raise RuntimeError("Dataset belum lolos QC.")

data_final.to_csv(OUTPUT_MAIN, index=False, encoding="utf-8-sig")
qc_final.to_csv(OUTPUT_QC, index=False, encoding="utf-8-sig")
qc_sample.to_csv(OUTPUT_SAMPLE, index=False, encoding="utf-8-sig")

print(f"[OK] CSV utama: {OUTPUT_MAIN}")
print(f"[OK] CSV QC matematis: {OUTPUT_QC}")
print(f"[OK] Sampel QC manual: {OUTPUT_SAMPLE}")

try:
    from google.colab import files

    files.download(OUTPUT_MAIN)
    files.download(OUTPUT_QC)
    files.download(OUTPUT_SAMPLE)
except ImportError:
    print("[INFO] Tombol download hanya tersedia di Google Colab.")

[OK] CSV utama: mpob_luas_tertanam_2023_2025.csv
[OK] CSV QC matematis: mpob_luas_tertanam_qc_2023_2025.csv
[OK] Sampel QC manual: mpob_luas_tertanam_sampel_qc_2023_2025.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Kirim ke satu Google Sheet dan bagikan akses Editor

In [15]:
from google.colab import auth

print("[INFO] Memulai autentikasi Google...")
auth.authenticate_user()
print("[OK] Autentikasi Google berhasil.")

[INFO] Memulai autentikasi Google...
[OK] Autentikasi Google berhasil.


In [16]:
GSHEET_URL = ""

if AUTO_UPLOAD_TO_GSHEET:
    if not DATASET_LOLOS_QC:
        raise RuntimeError("Upload dibatalkan karena dataset belum lolos QC.")

    from google.colab import auth
    import google.auth
    import gspread
    from googleapiclient.discovery import build

    auth.authenticate_user()
    credentials, _ = google.auth.default()
    gc = gspread.authorize(credentials)

    existing_books = gc.openall(SPREADSHEET_NAME)
    spreadsheet = existing_books[0] if existing_books else gc.create(SPREADSHEET_NAME)

    try:
        worksheet = spreadsheet.worksheet(WORKSHEET_NAME)
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=WORKSHEET_NAME,
            rows=max(len(data_final) + 50, 200),
            cols=max(len(data_final.columns) + 2, 12),
        )

    # Pertahankan revisi lama sesuai panduan IPOSS. Baris identik tidak digandakan;
    # perubahan nilai pada kunci yang sama ditambahkan sebagai baris revisi baru.
    existing_values = worksheet.get_all_records() if worksheet.get_all_values() else []
    if existing_values:
        existing_df = pd.DataFrame(existing_values)
        if set(FINAL_COLUMNS).issubset(existing_df.columns):
            existing_df = existing_df[FINAL_COLUMNS].copy()
            existing_df["year"] = pd.to_numeric(existing_df["year"], errors="coerce")
            existing_df["area_hectares"] = pd.to_numeric(
                existing_df["area_hectares"], errors="coerce"
            )
        else:
            raise RuntimeError(
                "Google Sheet lama memiliki struktur kolom berbeda. "
                "Ubah nama spreadsheet atau perbaiki kolom sebelum melanjutkan."
            )
    else:
        existing_df = pd.DataFrame(columns=FINAL_COLUMNS)

    rows_to_add = []
    for _, new_row in data_final.iterrows():
        if existing_df.empty:
            rows_to_add.append(new_row.to_dict())
            continue

        same_key = pd.Series(True, index=existing_df.index)
        for key in BUSINESS_KEY:
            same_key &= existing_df[key].astype(str).eq(str(new_row[key]))

        same_value = same_key & np.isclose(
            pd.to_numeric(existing_df["area_hectares"], errors="coerce"),
            float(new_row["area_hectares"]),
            equal_nan=True,
        )
        if same_value.any():
            continue

        row_dict = new_row.to_dict()
        if same_key.any():
            revision_note = "Revisi angka dari sumber MPOB; baris lama dipertahankan."
            row_dict["catatan"] = clean_text(
                f"{row_dict.get('catatan', '')} {revision_note}"
            )
        rows_to_add.append(row_dict)

    upload_df = pd.concat(
        [existing_df, pd.DataFrame(rows_to_add, columns=FINAL_COLUMNS)],
        ignore_index=True,
    )
    upload_df = upload_df[FINAL_COLUMNS]

    worksheet.clear()
    values = [FINAL_COLUMNS] + upload_df.where(upload_df.notna(), "").values.tolist()
    worksheet.update(values=values, range_name="A1", value_input_option="RAW")
    worksheet.freeze(rows=1)

    # Satu dataset hanya memakai satu worksheet data; hapus Sheet1 kosong bawaan.
    for ws in spreadsheet.worksheets():
        if ws.title == "Sheet1" and ws.title != WORKSHEET_NAME and not ws.get_all_values():
            spreadsheet.del_worksheet(ws)

    share_ok = False
    share_error = ""
    try:
        drive = build("drive", "v3", credentials=credentials, cache_discovery=False)
        permissions = drive.permissions().list(
            fileId=spreadsheet.id,
            fields="permissions(id,emailAddress,role,type)",
        ).execute().get("permissions", [])
        already_editor = any(
            permission.get("emailAddress", "").casefold() == SHARE_EDITOR_EMAIL.casefold()
            and permission.get("role") in {"writer", "owner"}
            for permission in permissions
        )
        if not already_editor:
            drive.permissions().create(
                fileId=spreadsheet.id,
                body={
                    "type": "user",
                    "role": "writer",
                    "emailAddress": SHARE_EDITOR_EMAIL,
                },
                sendNotificationEmail=True,
                fields="id",
            ).execute()
        share_ok = True
    except Exception as exc:
        share_error = str(exc)

    GSHEET_URL = spreadsheet.url
    print(f"[OK] Google Sheet: {GSHEET_URL}")
    print(f"[OK] Baris baru/revisi yang ditambahkan: {len(rows_to_add):,}")
    print(f"[OK] Total baris yang tersimpan di Sheet: {len(upload_df):,}")
    if share_ok:
        print(f"[OK] Akses Editor terverifikasi untuk {SHARE_EDITOR_EMAIL}")
    else:
        print(
            f"[WARNING] Pembagian otomatis gagal: {share_error}\n"
            f"Bagikan akses Editor secara manual kepada {SHARE_EDITOR_EMAIL}."
        )
else:
    print("[INFO] Upload Google Sheet dilewati karena AUTO_UPLOAD_TO_GSHEET=False.")

[OK] Google Sheet: https://docs.google.com/spreadsheets/d/1u7GWg6uxrap7gcc6hMCVtdtWpGv-NM7jLHV80fsNZSk
[OK] Baris baru/revisi yang ditambahkan: 117
[OK] Total baris yang tersimpan di Sheet: 117
[OK] Akses Editor terverifikasi untuk dataiposs@gmail.com


## 10. Ringkasan akhir dan sampel QC manual

In [17]:
print("=== RINGKASAN AKHIR ===")
print(f"Rentang tahun          : {data_final['year'].min()}-{data_final['year'].max()}")
print(f"Jumlah baris snapshot  : {len(data_final):,}")
print(f"Duplikat kunci         : {data_final.duplicated(BUSINESS_KEY).sum()}")
print(f"Nilai area kosong      : {data_final['area_hectares'].isna().sum()}")
print(f"QC negeri gagal        : {state_pivot['qc_status'].eq('FAIL').sum()}")
print(f"CSV utama              : {OUTPUT_MAIN}")
print(f"CSV QC                 : {OUTPUT_QC}")
print(f"Sampel QC manual       : {OUTPUT_SAMPLE}")
print(f"Google Sheet           : {GSHEET_URL or 'belum dibuat'}")

print("\nSampel 10 baris untuk dicocokkan dengan sumber/dasbor:")
display(qc_sample)

print("\nLog scraping:")
for line in SCRAPE_LOG:
    print(line)

print("\nLANGKAH TERAKHIR IPOSS:")
print("1. Pastikan dataiposs@gmail.com memiliki akses Editor.")
print("2. Tempel tautan Google Sheet ke kolom Link GSheet pada papan tugas.")
print("3. Isi tanggal mulai/selesai dan ubah status tugas sesuai progres.")
print("4. Setelah data tampil di dasbor, cocokkan 10 baris sampel dengan sumber MPOB.")

=== RINGKASAN AKHIR ===
Rentang tahun          : 2023-2025
Jumlah baris snapshot  : 117
Duplikat kunci         : 0
Nilai area kosong      : 0
QC negeri gagal        : 0
CSV utama              : mpob_luas_tertanam_2023_2025.csv
CSV QC                 : mpob_luas_tertanam_qc_2023_2025.csv
Sampel QC manual       : mpob_luas_tertanam_sampel_qc_2023_2025.csv
Google Sheet           : https://docs.google.com/spreadsheets/d/1u7GWg6uxrap7gcc6hMCVtdtWpGv-NM7jLHV80fsNZSk

Sampel 10 baris untuk dicocokkan dengan sumber/dasbor:


,year,state,area_type,area_hectares,sumber_link,input_at
0,2023,Melaka,matured,47667,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
1,2024,Negeri Sembilan,matured,168021,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
2,2024,Perak,immature,30460,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
3,2024,Selangor,total_planted,102980,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
4,2024,Terengganu,immature,21711,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
5,2025,Johor,total_planted,675558,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
6,2025,Perlis,total_planted,927,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
7,2025,Sabah,immature,228559,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
8,2025,Sabah,matured,1267999,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17
9,2025,Terengganu,immature,20124,https://prestasisawit.mpob.gov.my/en/plantatio...,2026-09-17



Log scraping:
[OK] Tahun 2023 berhasil dipilih.
[OK] Area by State 2023: 48 baris format panjang dari 16 baris tabel sumber.
[OK] Tahun 2024 berhasil dipilih.
[OK] Area by State 2024: 48 baris format panjang dari 16 baris tabel sumber.
[OK] Tahun 2025 berhasil dipilih.
[OK] Area by State 2025: 48 baris format panjang dari 16 baris tabel sumber.

LANGKAH TERAKHIR IPOSS:
1. Pastikan dataiposs@gmail.com memiliki akses Editor.
2. Tempel tautan Google Sheet ke kolom Link GSheet pada papan tugas.
3. Isi tanggal mulai/selesai dan ubah status tugas sesuai progres.
4. Setelah data tampil di dasbor, cocokkan 10 baris sampel dengan sumber MPOB.
